# Ders 10 - Üretimde AI Ajanları

Bu derste Microsoft Agent Framework (Python) kullanarak AI ajanları için **üretim kalıplarını** öğreneceksiniz. Kapsanan konular:

- **Gözlemlenebilirlik** — ajan etkileşimlerine zamanlama ve kayıt eklemek
- **Değerlendirme** — yanıt kalitesini puanlamak için bir değerlendirici ajan kullanmak
- **Maliyet yönetimi** — token optimizasyonu ve model seçimi stratejileri

Senaryo, kullanıcıların seyahat planlamasına yardımcı olan, üzerine izleme ve değerlendirme eklenmiş bir **seyahat acentesi**dir.


## Kurulum


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Üretim Dikkat Edilmesi Gerekenler

AI ajanlarını prototiplerden üretime taşımak, üç temel unsura titizlikle dikkat etmeyi gerektirir:

1. **Gözlemlenebilirlik** — Ajanın ne yaptığını, ne kadar sürdüğünü ve hangi araçları çağırdığını görmeniz gerekir. İzleme ve kayıt olmadan, üretim problemlerini hata ayıklamak neredeyse imkansızdır.

2. **Değerlendirme** — Otomatik kalite kontrolleri, ajan yanıtlarının zamanla doğru, eksiksiz ve yardımcı kalmasını sağlar. Bir değerlendirme ajanı, yanıtları tanımlı kriterlere göre puanlayabilir.

3. **Maliyet Yönetimi** — Token kullanımı doğrudan maliyeti etkiler. İstem optimizasyonu, model seçimi ve önbellekleme gibi stratejiler, kaliteyi feda etmeden giderleri kontrol altında tutmaya yardımcı olur.


## Gözlemlenebilir Bir Ajan İnşa Etmek

Seyahat araçları tanımlıyoruz ve ajan çağrısını zamanlama ile sarıyoruz, böylece gecikmeyi izleyebiliyoruz. Üretimde, OpenTelemetry veya benzeri bir izleme altyapısı ile entegre edersiniz.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Değerlendirme Kalıpları

Yaygın bir üretim kalıbı, ikinci bir ajanın **değerlendirici** olarak kullanılmasıdır. Değerlendirici, önceden tanımlanmış tamlık, doğruluk ve faydalılık gibi kriterlere göre birincil ajanın yanıtını puanlar.

Bu şunları sağlar:
- Yanıtlar kullanıcılara ulaşmadan önce otomatik kalite kapıları
- İstekte veya modelde değişiklik olduğunda regresyon tespiti
- Zaman içinde ajan performansının sürekli izlenmesi


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Cost Management Strategies

Controlling costs is critical for production AI agents. Here are key strategies:

| Strategy | Description |
|---|---|
| **Prompt optimization** | Keep system instructions concise. Remove redundant context to reduce input tokens. |
    "| **Model selection** | Use smaller, cheaper models (e.g. GPT-4.1-mini) for simple tasks like classification or extraction, and reserve larger models for complex reasoning. |\n",
| **Caching** | Cache tool results and frequent queries to avoid redundant API calls. |
| **Token budgets** | Set `max_tokens` limits to prevent unexpectedly long responses. |
| **Batching** | Group multiple user queries into a single API call where possible. |

In practice, a tiered approach works well: route straightforward requests to a fast, inexpensive model and escalate only complex queries to a more capable one.


## Özet

Bu derste şunları öğrendiniz:

1. **Gözlemlenebilirlik eklemek**: Zamanlama ve günlük kaydı ile ajan etkileşimlerine gözlemlenebilirlik ekleyerek izleme ve takip için temel oluşturmak.
2. **Ajan yanıtlarını değerlendirmek**: Tamlık, doğruluk ve yardımseverliği puanlayan bir değerlendirme ajanı kullanarak yanıtları otomatik olarak değerlendirmek.
3. **Maliyetleri yönetmek**: İstek optimizasyonu, model seçimi, önbellekleme ve token bütçeleri ile maliyetleri kontrol altında tutmak.

Bu üretim modelleri, AI ajanlarınızın ölçekli olarak güvenilir, ölçülebilir ve maliyet etkin olmasını sağlar.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba sarf etsek de, otomatik çevirilerin hata veya yanlışlık içerebileceğini lütfen unutmayınız. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek yanlış anlamalardan veya yanlış yorumlamalardan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
